# Ecualizador Gráfico por Tercios de Octava con Visualización FFT

Aplica filtros Butterworth de paso de banda sobre 32 bandas de tercio de octava (20 Hz–20 kHz).
Muestra en tiempo real la comparación FFT y la forma de onda original vs. filtrada.

| Componente | Detalle |
|------------|---------|
| Bandas | 32 tercios de octava (ISO 266): 20 Hz → 20 kHz |
| Filtros | Butterworth orden 4, salida SOS (numericamente estable) |
| Ganancia por banda | ±30 dB por fader |
| Visualización | FFT logarítmica + forma de onda en tiempo |

In [ ]:
import tkinter as tk
from tkinter import Scale, Label, Button, filedialog
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import numpy as np
from scipy.signal import butter, sosfilt
from scipy.fft import fft, fftfreq
from scipy.io import wavfile

# ─── Variables globales de estado ────────────────────────────────────────────
fs = None  # Frecuencia de muestreo del archivo cargado
x  = None  # Señal de audio (float32, mono)

# Frecuencias centrales de tercio de octava (ISO 266, 32 bandas)
fto_nom = np.array([
     20,   25,  31.5,
     40,   50,    63,
     80,  100,   125,
    160,  200,   250,
    315,  400,   500,
    630,  800,  1000,
   1250, 1600,  2000,
   2500, 3150,  4000,
   5000, 6300,  8000,
  10000,12500, 16000,
  18000,20000,
])

fader_valores = [0.0] * len(fto_nom)  # Ganancias por banda en dB (inicio: 0 dB)
# ─────────────────────────────────────────────────────────────────────────────


def fto_lim(fto_nom):
    """Límites de banda: ±1/6 de octava respecto a la frecuencia central."""
    return fto_nom * 2**(-1/6), fto_nom * 2**(1/6)


def but_pb(inf, sup, fs, order=4):
    """Coeficientes SOS de filtro Butterworth pasa-banda."""
    nyq  = 0.5 * fs
    low  = inf / nyq
    high = sup / nyq
    if low <= 0 or high >= 1:
        return None  # Banda fuera del rango de Nyquist → omitir
    return butter(order, [low, high], btype='band', output='sos')


def but_pb_filt(x, inf, sup, fs, order=4):
    """Filtra la señal con el filtro Butterworth de la banda [inf, sup]."""
    sos = but_pb(inf, sup, fs, order)
    if sos is None:
        return np.zeros_like(x)
    return sosfilt(sos, x)


def db_to_gain(db_value):
    return 10 ** (db_value / 20)


def filt_to(x, fto_nom, gain_values_db):
    """Aplica filtrado por bandas y escala cada banda según su ganancia en dB."""
    fto_inf, fto_sup = fto_lim(fto_nom)
    y = np.zeros((len(fto_nom), len(x)))
    for i in range(len(fto_nom)):
        banda = but_pb_filt(x, fto_inf[i], fto_sup[i], fs)
        y[i]  = banda * db_to_gain(gain_values_db[i])
    return y


def plot_fft_and_time_comparison(original_signal, filtered_signal, fs):
    N      = len(original_signal)
    freqs  = fftfreq(N, 1 / fs)
    eps    = 1e-10

    orig_mag   = np.abs(fft(original_signal))
    filt_mag   = np.abs(fft(filtered_signal))

    xticks     = [31.5, 63, 125, 250, 500, 1000, 4000, 8000, 16000]
    xtick_str  = ['31.5', '63', '125', '250', '500', '1k', '4k', '8k', '16k']

    ax1.clear()
    ax1.semilogx(freqs[:N//2], 20*np.log10(orig_mag[:N//2] + eps), label="Original", alpha=0.7)
    ax1.semilogx(freqs[:N//2], 20*np.log10(filt_mag[:N//2] + eps), label="Filtrada",  alpha=0.7)
    ax1.set_xticks(xticks)
    ax1.set_xticklabels(xtick_str)
    ax1.set_title("FFT: Señal Original vs. Filtrada (escala logarítmica)")
    ax1.set_xlabel("Frecuencia [Hz]")
    ax1.set_ylabel("Magnitud [dB]")
    ax1.set_xlim([20, 20000])
    ax1.legend()
    ax1.grid(True)

    t = np.arange(len(original_signal)) / fs
    ax2.clear()
    ax2.plot(t, original_signal, label="Original", alpha=0.7)
    ax2.plot(t, filtered_signal, label="Filtrada",  alpha=0.7)
    ax2.set_title("Forma de Onda: Original vs. Filtrada")
    ax2.set_xlabel("Tiempo [s]")
    ax2.set_ylabel("Amplitud")
    ax2.legend()
    ax2.grid(True)

    fig.tight_layout()
    canvas.draw()


def actualizar_grafica():
    if fs is None or x is None:
        return
    y = filt_to(x, fto_nom, fader_valores)
    plot_fft_and_time_comparison(x, np.sum(y, axis=0), fs)


def cargar_archivo_wav():
    global fs, x
    archivo_wav = filedialog.askopenfilename(filetypes=[("Archivo WAV", "*.wav")])
    if archivo_wav:
        fs, x = wavfile.read(archivo_wav)
        x = x.astype(np.float32)
        if x.ndim > 1:
            x = x[:, 0]  # Estéreo → usar canal izquierdo
        actualizar_grafica()


# ─── Interfaz gráfica ─────────────────────────────────────────────────────────
ventana = tk.Tk()
ventana.title("Ecualizador Gráfico — Tercios de Octava + FFT")
ventana.geometry("900x750")

Button(ventana, text="Cargar WAV", command=cargar_archivo_wav).pack(side=tk.TOP, pady=10)

frame_faders = tk.Frame(ventana)
frame_faders.pack(side=tk.LEFT, padx=10, pady=10)

for i in range(16):
    for col_offset, band_idx in enumerate([i * 2, i * 2 + 1]):
        fader = Scale(frame_faders, from_=-30, to=30, orient=tk.HORIZONTAL)
        fader.grid(row=i, column=col_offset * 2)
        Label(frame_faders, text=f"{int(fto_nom[band_idx])} Hz").grid(row=i, column=col_offset * 2 + 1)

        def make_callback(idx, fdr):
            def cb(event):
                fader_valores[idx] = fdr.get()
                actualizar_grafica()
            return cb

        fader.bind("<Motion>", make_callback(band_idx, fader))

frame_grafica = tk.Frame(ventana)
frame_grafica.pack(side=tk.RIGHT, padx=10, pady=10, expand=True, fill=tk.BOTH)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 8))
canvas = FigureCanvasTkAgg(fig, master=frame_grafica)
canvas.draw()
canvas.get_tk_widget().pack(side=tk.TOP, fill=tk.BOTH, expand=True)

ventana.mainloop()
